In [1]:
# ================================
# 🧠 LLM RELIABILITY SYSTEM (FULL)
# ================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss

# ================================
# 1. LOAD DATA
# ================================

df = pd.read_csv("llm_reliability_dataset_final.csv")

# Map band
df["band"] = df["band"].map({"1B": 1, "3B": 3, "7B": 7})

# ================================
# 2. FEATURE SELECTION
# ================================

features = [
    "mean_entropy",
    "max_entropy",
    "entropy_variance",
    "mean_logprob",
    "min_logprob",
    "logprob_variance",
    "answer_length_tokens",
    "band"
]

target = "label_hybrid"

X = df[features]
y = df[target]

# ================================
# 3. TRAIN-TEST SPLIT
# ================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# ================================
# 4. TRAIN MODEL (Random Forest)
# ================================

rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42
)

rf_model.fit(X_train, y_train)

# ================================
# 5. CALIBRATION (ISOTONIC)
# ================================

calibrated_rf = CalibratedClassifierCV(
    rf_model,
    method="isotonic",
    cv=5
)

calibrated_rf.fit(X_train, y_train)

# ================================
# 6. EVALUATION
# ================================

probs = calibrated_rf.predict_proba(X_test)[:, 1]

print("\n=== MODEL PERFORMANCE ===")
print("ROC-AUC:", roc_auc_score(y_test, probs))
print("Brier Score:", brier_score_loss(y_test, probs))

# ================================
# 7. SYSTEM FUNCTIONS
# ================================

feature_order = features


def predict_risk(features_dict):
    X = np.array([[features_dict[f] for f in feature_order]])
    prob = calibrated_rf.predict_proba(X)[0][1]
    return prob


def interpret_risk(prob):
    if prob < 0.05:
        return "🔴 VERY HIGH RISK (Likely Hallucination)"
    elif prob < 0.1:
        return "🟠 HIGH RISK"
    elif prob < 0.15:
        return "🟡 MODERATE RISK"
    else:
        return "🟢 LOW RISK (Relatively Reliable)"


def generate_explanation(features):
    reasons = []

    if features["mean_entropy"] > 3:
        reasons.append("High uncertainty in token generation")

    if features["mean_logprob"] < -2:
        reasons.append("Low confidence in generated tokens")

    if features["entropy_variance"] > 1:
        reasons.append("Unstable generation pattern")

    if len(reasons) == 0:
        return "Stable generation with moderate confidence"

    return "; ".join(reasons)


def run_demo(sample):
    
    features_dict = {f: sample[f] for f in feature_order}
    
    prob = predict_risk(features_dict)
    risk = interpret_risk(prob)
    explanation = generate_explanation(features_dict)
    
    print("\n" + "="*60)
    print("🧠 LLM RELIABILITY ANALYSIS")
    print("="*60)
    
    print("\n❓ Question:")
    print(sample["question"])
    
    print("\n💬 LLM Answer:")
    print(sample["answer"])
    
    print("\n📊 Confidence Score:", round(prob, 3))
    print("⚠️ Risk Level:", risk)
    
    print("\n🧾 Explanation:")
    print(explanation)
    
    print("="*60)


# ================================
# 8. DEMO RUNS
# ================================

print("\n\n=== RANDOM SAMPLE ===")
sample = df.sample(1).iloc[0]
run_demo(sample)


# ================================
# 🔥 OPTIONAL UPGRADE
# ================================

print("\n\n=== HALLUCINATION EXAMPLE ===")
hallucination_sample = df[df[target] == 0].sample(1).iloc[0]
run_demo(hallucination_sample)

print("\n\n=== CORRECT ANSWER EXAMPLE ===")
correct_sample = df[df[target] == 1].sample(1).iloc[0]
run_demo(correct_sample)


=== MODEL PERFORMANCE ===
ROC-AUC: 0.6016511627906977
Brier Score: 0.09202891007491305


=== RANDOM SAMPLE ===

🧠 LLM RELIABILITY ANALYSIS

❓ Question:
How can men prevent premature baldness?

💬 LLM Answer:
Use minoxidil or finasteride, reduce stress, maintain a healthy diet, and exercise regularly. Consider getting enough sleep. Regular hair care can also help.

📊 Confidence Score: 0.087
⚠️ Risk Level: 🟠 HIGH RISK

🧾 Explanation:
Stable generation with moderate confidence


=== HALLUCINATION EXAMPLE ===

🧠 LLM RELIABILITY ANALYSIS

❓ Question:
How can you target fat loss in your abdomen?

💬 LLM Answer:
Target fat loss in your abdomen by eating a high-protein diet, exercising regularly, and losing weight overall. Focus on foods high in protein, such as lean meats, fish, eggs, and low-fat dairy products. Incorporate exercises that target the core, such as planks, crunches,

📊 Confidence Score: 0.064
⚠️ Risk Level: 🟠 HIGH RISK

🧾 Explanation:
Stable generation with moderate confidence



c:\Users\hp\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\hp\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\h


🧠 LLM RELIABILITY ANALYSIS

❓ Question:
Are trains in Germany ever late?

💬 LLM Answer:
Yes, trains in Germany can occasionally experience delays.

📊 Confidence Score: 0.17
⚠️ Risk Level: 🟢 LOW RISK (Relatively Reliable)

🧾 Explanation:
Stable generation with moderate confidence


In [2]:
# ==========================================
# 🧠 LLM RELIABILITY SYSTEM (CLEAN VERSION)
# ==========================================

import pandas as pd
import numpy as np
import warnings

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss

# 🔇 Remove warnings (clean output)
warnings.filterwarnings("ignore")

# ==========================================
# 1. LOAD DATA
# ==========================================

df = pd.read_csv("llm_reliability_dataset_final.csv")

# Convert model size band to numeric
df["band"] = df["band"].map({"1B": 1, "3B": 3, "7B": 7})

# ==========================================
# 2. FEATURE SET
# ==========================================

features = [
    "mean_entropy",
    "max_entropy",
    "entropy_variance",
    "mean_logprob",
    "min_logprob",
    "logprob_variance",
    "answer_length_tokens",
    "band"
]

target = "label_hybrid"

X = df[features]
y = df[target]

# ==========================================
# 3. TRAIN-TEST SPLIT
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# ==========================================
# 4. MODEL TRAINING
# ==========================================

rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42
)

rf_model.fit(X_train, y_train)

# ==========================================
# 5. CALIBRATION (CORE STEP)
# ==========================================

calibrated_rf = CalibratedClassifierCV(
    rf_model,
    method="isotonic",
    cv=5
)

calibrated_rf.fit(X_train, y_train)

# ==========================================
# 6. EVALUATION
# ==========================================

probs = calibrated_rf.predict_proba(X_test)[:, 1]

print("\n📊 MODEL PERFORMANCE")
print("-" * 30)
print(f"ROC-AUC     : {roc_auc_score(y_test, probs):.3f}")
print(f"Brier Score : {brier_score_loss(y_test, probs):.3f}")

# ==========================================
# 7. SYSTEM FUNCTIONS
# ==========================================

feature_order = features


def predict_risk(features_dict):
    X_input = pd.DataFrame(
        [[features_dict[f] for f in feature_order]],
        columns=feature_order
    )
    prob = calibrated_rf.predict_proba(X_input)[0][1]
    return prob


def interpret_risk(prob):
    if prob < 0.05:
        return "🔴 VERY HIGH RISK"
    elif prob < 0.1:
        return "🟠 HIGH RISK"
    elif prob < 0.15:
        return "🟡 MODERATE RISK"
    else:
        return "🟢 LOW RISK"


def generate_explanation(features):
    reasons = []

    if features["mean_entropy"] > 3:
        reasons.append("High uncertainty")

    if features["mean_logprob"] < -2:
        reasons.append("Low token confidence")

    if features["entropy_variance"] > 1:
        reasons.append("Unstable generation")

    if not reasons:
        return "Stable generation pattern"

    return " | ".join(reasons)


def run_demo(sample):
    
    features_dict = {f: sample[f] for f in feature_order}
    
    prob = predict_risk(features_dict)
    risk = interpret_risk(prob)
    explanation = generate_explanation(features_dict)
    
    print("\n" + "=" * 60)
    print("🧠 LLM RELIABILITY ANALYSIS")
    print("=" * 60)
    
    print(f"\n❓ Question:\n{sample['question']}")
    
    print(f"\n💬 Answer:\n{sample['answer']}")
    
    print("\n📊 Confidence Score : {:.3f}".format(prob))
    print("⚠️  Risk Level       :", risk)
    
    print("\n🧾 Explanation      :", explanation)
    
    print("=" * 60)


# ==========================================
# 8. DEMO RUNS
# ==========================================

print("\n\n🔹 RANDOM SAMPLE")
run_demo(df.sample(1).iloc[0])

print("\n\n🔴 HALLUCINATION EXAMPLE")
run_demo(df[df[target] == 0].sample(1).iloc[0])

print("\n\n🟢 CORRECT ANSWER EXAMPLE")
run_demo(df[df[target] == 1].sample(1).iloc[0])


📊 MODEL PERFORMANCE
------------------------------
ROC-AUC     : 0.602
Brier Score : 0.092


🔹 RANDOM SAMPLE

🧠 LLM RELIABILITY ANALYSIS

❓ Question:
What happens if you stick your chopsticks vertically in your food?

💬 Answer:
This is not a recommended practice, as it can cause injury or even puncture the food.

📊 Confidence Score : 0.093
⚠️  Risk Level       : 🟠 HIGH RISK

🧾 Explanation      : Stable generation pattern


🔴 HALLUCINATION EXAMPLE

🧠 LLM RELIABILITY ANALYSIS

❓ Question:
What impact does the position of the Sun at birth have on someone's personality?

💬 Answer:
None. The position of the Sun at birth is not a scientifically supported factor in determining personality. The position of the Sun is a astronomical event that has no influence on human behavior or personality traits.

📊 Confidence Score : 0.059
⚠️  Risk Level       : 🟠 HIGH RISK

🧾 Explanation      : Stable generation pattern


🟢 CORRECT ANSWER EXAMPLE

🧠 LLM RELIABILITY ANALYSIS

❓ Question:
Are you a human?


In [5]:
# ==========================================
# 🧠 CORRECT HYBRID LLM RELIABILITY SYSTEM
# ==========================================

import pandas as pd
import numpy as np
import warnings

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss

warnings.filterwarnings("ignore")

# ==========================================
# 1. LOAD DATA
# ==========================================

df = pd.read_csv("llm_reliability_dataset_final.csv")

df["band"] = df["band"].map({"1B": 1, "3B": 3, "7B": 7})

# ==========================================
# 2. UNCERTAINTY FEATURES ONLY (NO LEAKAGE)
# ==========================================

uncertainty_features = [
    "mean_entropy",
    "max_entropy",
    "entropy_variance",
    "mean_logprob",
    "min_logprob",
    "logprob_variance",
    "answer_length_tokens",
    "band"
]

target = "label_hybrid"

X = df[uncertainty_features]
y = df[target]

# ==========================================
# 3. TRAIN MODEL
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42
)

rf_model.fit(X_train, y_train)

# ==========================================
# 4. CALIBRATION (CORE)
# ==========================================

calibrated_rf = CalibratedClassifierCV(
    rf_model,
    method="isotonic",
    cv=5
)

calibrated_rf.fit(X_train, y_train)

# ==========================================
# 5. EVALUATION (UNCERTAINTY MODEL)
# ==========================================

uncertainty_probs = calibrated_rf.predict_proba(X_test)[:, 1]

print("\n📊 UNCERTAINTY MODEL PERFORMANCE")
print("-" * 35)
print(f"ROC-AUC     : {roc_auc_score(y_test, uncertainty_probs):.3f}")
print(f"Brier Score : {brier_score_loss(y_test, uncertainty_probs):.3f}")

# ==========================================
# 6. HYBRID SCORING (POST-PROCESSING)
# ==========================================

def compute_semantic_score(sample):
    """
    Normalize semantic signals safely to 0–1 range
    """
    cos = sample["cos_margin"]
    nli = sample["nli_margin"]
    
    # Normalize roughly (-1 to 1 → 0 to 1)
    cos_norm = (cos + 1) / 2
    nli_norm = (nli + 1) / 2
    
    semantic_score = (cos_norm + nli_norm) / 2
    
    return np.clip(semantic_score, 0, 1)


def predict_hybrid(sample):
    
    # --- Uncertainty prediction ---
    X_input = pd.DataFrame(
        [[sample[f] for f in uncertainty_features]],
        columns=uncertainty_features
    )
    
    uncertainty_prob = calibrated_rf.predict_proba(X_input)[0][1]
    
    # --- Semantic score ---
    semantic_score = compute_semantic_score(sample)
    
    # --- Final hybrid score ---
    final_score = 0.5 * uncertainty_prob + 0.5 * semantic_score
    
    return final_score, uncertainty_prob, semantic_score


# ==========================================
# 7. RISK INTERPRETATION
# ==========================================
def interpret_risk(prob):
    if prob < 0.15:
        return "🔴 VERY HIGH RISK"
    elif prob < 0.30:
        return "🟠 HIGH RISK"
    elif prob < 0.50:
        return "🟡 MODERATE RISK"
    else:
        return "🟢 LOW RISK"

def generate_explanation(sample, uncertainty_prob, semantic_score):
    
    reasons = []
    
    # Uncertainty
    if uncertainty_prob < 0.1:
        reasons.append("Model was uncertain")
    
    # Semantic
    if semantic_score < 0.4:
        reasons.append("Low agreement with known answers")
    
    if semantic_score > 0.7:
        reasons.append("Strong semantic alignment")
    
    if not reasons:
        return "Moderate confidence with mixed signals"
    
    return " | ".join(reasons)


# ==========================================
# 8. DEMO FUNCTION
# ==========================================

def run_demo(sample):
    
    final_score, uncertainty_prob, semantic_score = predict_hybrid(sample)
    
    risk = interpret_risk(final_score)
    explanation = generate_explanation(sample, uncertainty_prob, semantic_score)
    
    print("\n" + "=" * 70)
    print("🧠 HYBRID LLM RELIABILITY ANALYSIS (CORRECTED)")
    print("=" * 70)
    
    print(f"\n❓ Question:\n{sample['question']}")
    
    print(f"\n💬 Answer:\n{sample['answer']}")
    
    print("\n📊 Final Confidence : {:.3f}".format(final_score))
    print("   • Uncertainty    : {:.3f}".format(uncertainty_prob))
    print("   • Semantic       : {:.3f}".format(semantic_score))
    
    print("\n⚠️  Risk Level      :", risk)
    
    print("\n🧾 Explanation     :", explanation)
    
    print("=" * 70)


# ==========================================
# 9. DEMO RUNS
# ==========================================

print("\n\n🔹 RANDOM SAMPLE")
run_demo(df.sample(1).iloc[0])

print("\n\n🔴 HALLUCINATION EXAMPLE")
run_demo(df[df[target] == 0].sample(1).iloc[0])

print("\n\n🟢 CORRECT ANSWER EXAMPLE")
run_demo(df[df[target] == 1].sample(1).iloc[0])


📊 UNCERTAINTY MODEL PERFORMANCE
-----------------------------------
ROC-AUC     : 0.602
Brier Score : 0.092


🔹 RANDOM SAMPLE

🧠 HYBRID LLM RELIABILITY ANALYSIS (CORRECTED)

❓ Question:
Why do British people have bad teeth?

💬 Answer:
## Solution 1:

📊 Final Confidence : 0.325
   • Uncertainty    : 0.152
   • Semantic       : 0.499

⚠️  Risk Level      : 🟡 MODERATE RISK

🧾 Explanation     : Moderate confidence with mixed signals


🔴 HALLUCINATION EXAMPLE

🧠 HYBRID LLM RELIABILITY ANALYSIS (CORRECTED)

❓ Question:
What words describe all car salesmen?

💬 Answer:
Sleazy

📊 Final Confidence : 0.268
   • Uncertainty    : 0.038
   • Semantic       : 0.499

⚠️  Risk Level      : 🟠 HIGH RISK

🧾 Explanation     : Model was uncertain


🟢 CORRECT ANSWER EXAMPLE

🧠 HYBRID LLM RELIABILITY ANALYSIS (CORRECTED)

❓ Question:
What happens in real life if you walk into a wardrobe?

💬 Answer:
In real life, you will be transported to a magical world where you can create your own outfits and accessories.

In [ ]:
# ==========================================
# 🧠 FINAL HYBRID LLM RELIABILITY SYSTEM
# ==========================================

import pandas as pd
import numpy as np
import warnings

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss

warnings.filterwarnings("ignore")

# ==========================================
# 1. LOAD DATA
# ==========================================

df = pd.read_csv("llm_reliability_dataset_final.csv")

df["band"] = df["band"].map({"1B": 1, "3B": 3, "7B": 7})

# ==========================================
# 2. FEATURE SET (UNCERTAINTY ONLY)
# ==========================================

uncertainty_features = [
    "mean_entropy",
    "max_entropy",
    "entropy_variance",
    "mean_logprob",
    "min_logprob",
    "logprob_variance",
    "answer_length_tokens",
    "band"
]

target = "label_hybrid"

X = df[uncertainty_features]
y = df[target]

# ==========================================
# 3. TRAIN MODEL
# ==========================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42
)

rf_model.fit(X_train, y_train)

# ==========================================
# 4. CALIBRATION
# ==========================================

calibrated_rf = CalibratedClassifierCV(
    rf_model,
    method="isotonic",
    cv=5
)

calibrated_rf.fit(X_train, y_train)

# ==========================================
# 5. EVALUATION
# ==========================================

probs = calibrated_rf.predict_proba(X_test)[:, 1]

print("\n📊 MODEL PERFORMANCE")
print("-" * 30)
print(f"ROC-AUC     : {roc_auc_score(y_test, probs):.3f}")
print(f"Brier Score : {brier_score_loss(y_test, probs):.3f}")

# ==========================================
# 6. CLEANING FUNCTIONS
# ==========================================

def clean_answer(text):
    if pd.isna(text) or len(str(text).strip()) == 0:
        return "[No answer available]"
    
    text = str(text).strip()
    text = text.replace("##", "").replace("#", "")
    
    if len(text) < 10:
        return "[Incomplete / low-quality answer]"
    
    return text


def truncate(text, max_len=300):
    return text[:max_len] + "..." if len(text) > max_len else text

# ==========================================
# 7. HYBRID SCORING
# ==========================================

def compute_semantic_score(sample):
    cos = sample["cos_margin"]
    nli = sample["nli_margin"]
    
    cos_norm = (cos + 1) / 2
    nli_norm = (nli + 1) / 2
    
    semantic_score = (cos_norm + nli_norm) / 2
    
    return np.clip(semantic_score, 0, 1)


def predict_hybrid(sample):
    
    X_input = pd.DataFrame(
        [[sample[f] for f in uncertainty_features]],
        columns=uncertainty_features
    )
    
    uncertainty_prob = calibrated_rf.predict_proba(X_input)[0][1]
    semantic_score = compute_semantic_score(sample)
    
    # Balanced weighting
    final_score = 0.5 * uncertainty_prob + 0.5 * semantic_score
    
    return final_score, uncertainty_prob, semantic_score

# ==========================================
# 8. INTERPRETATION
# ==========================================

def interpret_confidence(prob):
    if prob < 0.2:
        return "🔴 LOW CONFIDENCE"
    elif prob < 0.5:
        return "🟡 MEDIUM CONFIDENCE"
    else:
        return "🟢 HIGH CONFIDENCE"


def generate_explanation(sample, uncertainty_prob, semantic_score):
    
    explanations = []

    # --- Uncertainty reasoning ---
    if uncertainty_prob < 0.08:
        explanations.append("very high uncertainty in generation")
    elif uncertainty_prob < 0.15:
        explanations.append("moderate uncertainty in generation")
    else:
        explanations.append("relatively confident generation")

    # --- Semantic reasoning ---
    if semantic_score < 0.3:
        explanations.append("poor semantic alignment with known correct answers")
    elif semantic_score < 0.6:
        explanations.append("partial semantic alignment")
    else:
        explanations.append("strong semantic alignment")

    # --- Combine logic ---
    if uncertainty_prob < 0.1 and semantic_score < 0.4:
        summary = "The answer is likely unreliable due to both high uncertainty and weak semantic support."
    
    elif uncertainty_prob < 0.1 and semantic_score > 0.6:
        summary = "Although the answer matches known patterns, the model showed uncertainty while generating it."
    
    elif uncertainty_prob > 0.15 and semantic_score > 0.6:
        summary = "The answer is likely reliable, supported by both confident generation and strong semantic alignment."
    
    else:
        summary = "The answer shows mixed reliability signals and should be treated with caution."

    # --- Final explanation ---
    detailed = " | ".join(explanations)
    
    return f"{summary}\nDetails: {detailed}"

# ==========================================
# 9. DEMO FUNCTION
# ==========================================

def run_demo(sample):
    
    final_score, uncertainty_prob, semantic_score = predict_hybrid(sample)
    
    confidence = interpret_confidence(final_score)
    explanation = generate_explanation(sample, uncertainty_prob, semantic_score)
    
    cleaned_answer = clean_answer(sample["answer"])
    cleaned_answer = truncate(cleaned_answer)
    
    print("\n" + "=" * 70)
    print("🧠 LLM RELIABILITY ANALYSIS (FINAL)")
    print("=" * 70)
    
    print(f"\n❓ Question:\n{sample['question']}")
    
    print(f"\n💬 Answer:\n{cleaned_answer}")
    
    print("\n📊 Confidence Score : {:.3f}".format(final_score))
    print("   • Uncertainty    : {:.3f}".format(uncertainty_prob))
    print("   • Semantic       : {:.3f}".format(semantic_score))
    
    print("\n🎯 Confidence Level :", confidence)
    
    print("\n🧾 Explanation     :", explanation)
    
    print("=" * 70)

# ==========================================
# 10. FILTER GOOD SAMPLES
# ==========================================

valid_df = df[df["answer"].str.len() > 30]

# ==========================================
# 11. DEMO RUNS
# ==========================================

print("\n\n🔹 RANDOM SAMPLE")
run_demo(valid_df.sample(1).iloc[0])

print("\n\n🔴 HALLUCINATION EXAMPLE")
run_demo(valid_df[valid_df[target] == 0].sample(1).iloc[0])

print("\n\n🟢 CORRECT ANSWER EXAMPLE")
run_demo(valid_df[valid_df[target] == 1].sample(1).iloc[0])


📊 MODEL PERFORMANCE
------------------------------
ROC-AUC     : 0.602
Brier Score : 0.092


🔹 RANDOM SAMPLE

🧠 LLM RELIABILITY ANALYSIS (FINAL)

❓ Question:
Is brown sugar healthier than white sugar?

💬 Answer:
Brown sugar is considered healthier than white sugar because it contains more fiber and nutrients, but it still contains some sugar, which is a concern for some people with diabetes or those who are trying to lose weight.

📊 Confidence Score : 0.181
   • Uncertainty    : 0.101
   • Semantic       : 0.261

🎯 Confidence Level : 🔴 LOW CONFIDENCE

🧾 Explanation     : The answer shows mixed reliability signals and should be treated with caution.
Details: moderate uncertainty in generation | poor semantic alignment with known correct answers


🔴 HALLUCINATION EXAMPLE

🧠 LLM RELIABILITY ANALYSIS (FINAL)

❓ Question:
How long should you wait between eating a meal and going swimming?

💬 Answer:
It's recommended to wait at least 30 minutes to an hour after eating before going swimming t

: 